# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：**学号：24012465，github:later-bin，姓名：舍滨**  
第5天专题（A/B/C/D/E）：**A**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。

## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "24012465"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))

## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [ ]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


In [ ]:
# TODO：填写4个业务问题和图表选择理由
business_questions = {
    "category_bar": "不同支付方式的用户规模和流失率是否存在差异？哪种支付方式的用户流失风险最高？",
    "behavior_scatter": "用户的订单数量与返现金额之间是否存在关联？流失用户和留存用户在订单-返现行为上有何不同？",
    "ordered_line": "随着用户生命周期阶段的推进（新用户→24个月以上），流失率呈现怎样的变化趋势？",
    "composition_chart": "用户在城市等级（1/2/3级）上的分布结构是怎样的？哪类城市用户占比最大？",
}

chart_reasons = {
    "category_bar": "柱状图适合比较5种离散支付方式之间的数值差异，使用柱高表示用户数并标注流失率，便于横向对比各支付方式用户的规模和流失风险。",
    "behavior_scatter": "散点图能够展示OrderCount和CashbackAmount两个连续变量的联合分布，通过颜色区分Churn，可直观观察流失与留存用户在行为空间中的聚集与分离模式。",
    "ordered_line": "TenureGroup具有明确的生命周期阶段顺序（从新用户到24个月以上），折线图适合展示流失率随阶段推进的单调变化趋势，清晰传达流失风险的阶段性特征。",
    "composition_chart": "CityTier共3个类别（≤5），适合使用环形图展示用户在城市等级上的整体构成占比，直观且信息密度适中。",
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")

## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
# TODO：完成绘图数据。使用PreferredPaymentMode作为主分组字段（5种支付方式，适合柱状图横向对比）。
category_field = "PreferredPaymentMode"
category_summary = None

category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

# 按流失率降序排列，便于观察高风险支付方式
category_summary = category_summary.sort_values("流失率", ascending=False).reset_index(drop=True)

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)

In [ ]:
# TODO：绘制并保存柱状图——展示不同支付方式的用户数和流失率
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

plot_data = category_summary

# 柱状图：展示用户数
colors = ["#E74C3C", "#E67E22", "#F1C40F", "#2ECC71", "#3498DB"]
bars = ax_bar.bar(
    plot_data[category_field],
    plot_data["用户数"],
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    width=0.6,
)

# 在每根柱子上标注流失率和样本量
for bar, (_, row) in zip(bars, plot_data.iterrows()):
    height = bar.get_height()
    ax_bar.text(
        bar.get_x() + bar.get_width() / 2,
        height + 15,
        f'流失率: {row["流失率"]:.1%}\n(n={int(row["用户数"])})',
        ha="center", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
    )

ax_bar.set_title("不同支付方式的用户数与流失率对比", fontsize=14, fontweight="bold")
ax_bar.set_xlabel("支付方式", fontsize=12)
ax_bar.set_ylabel("用户数", fontsize=12)
ax_bar.set_ylim(0, plot_data["用户数"].max() * 1.25)
ax_bar.tick_params(axis="x", rotation=15)

fig_bar.tight_layout()

bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))

### 柱状图结论

- 观察：不同支付方式的用户流失率存在明显差异。E wallet用户的流失率最高（约21.2%），而UPI用户的流失率最低（约13.8%），两者相差约7.4个百分点。Cash on Delivery用户的流失率也较高（约19.6%），紧追其后。
- 证据：Debit Card用户数最多（2,314人，流失率16.9%），Credit Card次之（1,774人，流失率15.3%）。E wallet用户仅614人但流失率达21.2%，UPI用户414人流失率13.8%。各分组样本量均在400人以上，统计上具有可观察性。
- 边界：该图只能展示支付方式与流失率的组间关联，不能证明使用某种支付方式会导致用户流失。支付方式偏好可能与用户收入、消费习惯等第三方因素相关，这些混淆变量未在图中控制。

## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
# TODO：选择两个数值字段——OrderCount与CashbackAmount，一行代表一个用户
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

# 按Churn分组绘制散点图
churn_colors = {0: "#2ECC71", 1: "#E74C3C"}
churn_labels = {0: "留存用户 (Churn=0)", 1: "流失用户 (Churn=1)"}

for churn_val in [0, 1]:
    subset = df[df["Churn"] == churn_val]
    ax_scatter.scatter(
        subset[x_field],
        subset[y_field],
        c=churn_colors[churn_val],
        label=churn_labels[churn_val],
        alpha=0.45,
        edgecolors="none",
        s=20,
    )

ax_scatter.set_title("用户订单数与返现金额散点图（按流失状态着色）", fontsize=14, fontweight="bold")
ax_scatter.set_xlabel("订单数 (OrderCount)", fontsize=12)
ax_scatter.set_ylabel("返现金额 (CashbackAmount, 元)", fontsize=12)
ax_scatter.legend(loc="upper left", fontsize=10, markerscale=2)
ax_scatter.grid(True, alpha=0.3)

# 添加参考说明
ax_scatter.text(
    0.95, 0.05,
    f"总用户数: {len(df)}, 流失: {df['Churn'].sum()}, 留存: {len(df) - df['Churn'].sum()}",
    transform=ax_scatter.transAxes, fontsize=9, ha="right",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)

fig_scatter.tight_layout()

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))

### 散点图结论

- 观察：流失用户（红色）主要集中在低订单数区域（OrderCount ≤ 3），其返现金额也相应较低。留存用户（绿色）在整个订单数范围内均有分布，且在OrderCount较高时（≥5），几乎全部为留存用户。返现金额与订单数整体呈正相关趋势。
- 证据：当OrderCount ≥ 5时，流失用户的数量极少（不足20人），而留存用户超过1,500人。低订单数区域（OrderCount ≤ 2）中流失用户密度明显更高，约占该区域总用户的30%以上。两个变量呈中等正相关，高订单数和高返现行为是用户留存的重要行为信号。
- 边界：相关关系不等于因果关系。高订单数可能导致高返现（因为返现与订单挂钩），但不能推断"增加返现就能降低流失"——留存用户可能本身就活跃，订单数和返现都是"活跃"的结果而非原因。此外，`CashbackAmount`是返现金额而非消费金额，不能用于GMV分析。

## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# TODO：准备有序绘图数据——使用TenureGroup作为有序阶段字段
ordered_field = "TenureGroup"
ordered_summary = None

ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

# 按生命周期阶段排序
ordered_summary[ordered_field] = pd.Categorical(
    ordered_summary[ordered_field],
    categories=TENURE_ORDER,
    ordered=True,
)
ordered_summary = ordered_summary.sort_values(ordered_field).reset_index(drop=True)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)

In [ ]:
# TODO：绘制折线图——展示TenureGroup各阶段流失率变化趋势，标注比例和样本量
fig_line, ax_line = plt.subplots(figsize=(10, 6))

plot_data = ordered_summary

# 绘制流失率折线
ax_line.plot(
    plot_data[ordered_field].astype(str),
    plot_data["流失率"],
    marker="o",
    markersize=10,
    linewidth=2.5,
    color="#E74C3C",
    markerfacecolor="#E74C3C",
    markeredgecolor="white",
    markeredgewidth=1.5,
)

# 在每个数据点标注流失率和样本量
for idx, row in plot_data.iterrows():
    ax_line.annotate(
        f'流失率: {row["流失率"]:.1%}\n(n={int(row["用户数"])})',
        (idx, row["流失率"]),
        textcoords="offset points",
        xytext=(0, 18),
        ha="center",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.85),
    )

# 添加总体流失率参考线
overall_churn = df["Churn"].mean()
ax_line.axhline(
    y=overall_churn, color="gray", linestyle="--", linewidth=1, alpha=0.7,
    label=f"总体流失率: {overall_churn:.1%}",
)

ax_line.set_title("不同生命周期阶段的用户流失率趋势", fontsize=14, fontweight="bold")
ax_line.set_xlabel("生命周期阶段（有序）", fontsize=12)
ax_line.set_ylabel("流失率", fontsize=12)
ax_line.set_ylim(0, plot_data["流失率"].max() * 1.3)
ax_line.legend(fontsize=10)
ax_line.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_line.grid(True, alpha=0.3, axis="y")

fig_line.tight_layout()

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))

### 折线图结论

- 观察：流失率随生命周期阶段推进呈严格单调递减趋势。新用户阶段流失率最高（53.5%），到24个月以上阶段降至0.0%，各阶段之间的流失率差距明显且持续收窄。
- 证据：新用户（508人）流失率53.5%，0-6个月（1,642人）降至25.9%，7-12个月（1,584人）降至9.8%，13-24个月（1,467人）降至6.5%，24个月以上（429人）降至0.0%。所有阶段均远低于或高于总体流失率参考线（16.8%），说明生命周期是流失的强关联因素。样本量均超过400人，结果具有统计可信度。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势。由于数据为横截面快照，不能直接证明"用户在平台停留时间越长则越不容易流失"的因果过程——这可能只是幸存者偏差：早期就流失的用户已被排除在老用户群体之外。高流失率的新用户群体可能包含大量一次性用户，其行为与长期用户有本质差异。

## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
# TODO：选择构成字段并准备汇总表——CityTier（3个类别，≤5，适合环形图）
composition_field = "CityTier"
composition_summary = None

composition_summary = (
    df.groupby(composition_field)
      .agg(用户数=("CustomerID", "nunique"))
      .reset_index()
)

# 计算占比
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()

# 添加城市等级标签便于阅读
tier_labels = {1: "一线城市", 2: "二线城市", 3: "三线城市"}
composition_summary["城市等级"] = composition_summary[composition_field].map(tier_labels)

# 按占比降序排列
composition_summary = composition_summary.sort_values("占比", ascending=False).reset_index(drop=True)

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)

In [ ]:
# TODO：类别不超过5个时绘制环形图（CityTier有3个类别）
fig_composition, ax_composition = plt.subplots(figsize=(10, 6))

plot_data = composition_summary

# 环形图配色
donut_colors = ["#3498DB", "#2ECC71", "#F39C12"]
explode = (0.03, 0.03, 0.03)

wedges, texts, autotexts = ax_composition.pie(
    plot_data["占比"],
    labels=plot_data["城市等级"],
    autopct="%1.1f%%",
    startangle=90,
    colors=donut_colors,
    explode=explode,
    pctdistance=0.78,
)

# 添加中心圆制造环形效果
centre_circle = plt.Circle((0, 0), 0.55, fc="white", linewidth=0)
ax_composition.add_artist(centre_circle)

# 优化文本样式
for text in texts:
    text.set_fontsize(12)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight("bold")
    autotext.set_color("white")

# 在中心添加总用户数
ax_composition.text(
    0, 0,
    f"总用户数\n{int(plot_data['用户数'].sum())}",
    ha="center", va="center", fontsize=13, fontweight="bold",
)

ax_composition.set_title("用户城市等级分布构成", fontsize=14, fontweight="bold")

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))

### 构成图结论

- 观察：平台用户主要集中在一线城市，占比超过六成，二三线城市用户合计不足四成。用户在城市等级上的分布呈现明显的"头部集中"特征。
- 证据：一线城市（CityTier=1）用户3,666人，占比65.1%；三线城市（CityTier=3）用户1,722人，占比30.6%；二线城市（CityTier=2）用户仅242人，占比4.3%。一线城市用户数量是二线城市的15倍以上。
- 边界：该图仅展示用户在城市等级上的静态构成，不能反映各城市等级用户的活跃度、流失率或消费能力差异。二线城市用户占比仅4.3%（242人），后续分析中作为独立分组时需注意样本量限制。此外，城市等级的划分标准和数据来源未明确，可能影响结论的外部有效性。

## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 子图1（左上）：支付方式柱状图 ---
ax1 = axes[0, 0]
bar_data = category_summary.sort_values("流失率", ascending=False)
bars = ax1.bar(
    bar_data[category_field], bar_data["用户数"],
    color=["#E74C3C", "#E67E22", "#F1C40F", "#2ECC71", "#3498DB"],
    edgecolor="white", linewidth=0.6, width=0.6,
)
for bar, (_, row) in zip(bars, bar_data.iterrows()):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
             f'{row["流失率"]:.1%}', ha="center", fontsize=8, fontweight="bold")
ax1.set_title("各支付方式用户数与流失率", fontsize=12, fontweight="bold")
ax1.set_ylabel("用户数")
ax1.set_ylim(0, bar_data["用户数"].max() * 1.2)
ax1.tick_params(axis="x", rotation=15, labelsize=8)

# --- 子图2（右上）：订单-返现散点图 ---
ax2 = axes[0, 1]
for churn_val, color, label in [(0, "#2ECC71", "留存"), (1, "#E74C3C", "流失")]:
    subset = df[df["Churn"] == churn_val]
    ax2.scatter(subset["OrderCount"], subset["CashbackAmount"],
                c=color, label=label, alpha=0.35, edgecolors="none", s=12)
ax2.set_title("订单数与返现金额散点图", fontsize=12, fontweight="bold")
ax2.set_xlabel("订单数")
ax2.set_ylabel("返现金额（元）")
ax2.legend(fontsize=8, markerscale=2)
ax2.grid(True, alpha=0.2)

# --- 子图3（左下）：生命周期折线图 ---
ax3 = axes[1, 0]
line_data = ordered_summary
ax3.plot(
    line_data[ordered_field].astype(str), line_data["流失率"],
    marker="o", markersize=8, linewidth=2, color="#E74C3C",
    markerfacecolor="#E74C3C", markeredgecolor="white", markeredgewidth=1.5,
)
overall = df["Churn"].mean()
ax3.axhline(y=overall, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
for idx, row in line_data.iterrows():
    ax3.annotate(f'{row["流失率"]:.1%}', (idx, row["流失率"]),
                 textcoords="offset points", xytext=(0, 12), ha="center", fontsize=8)
ax3.set_title("各生命周期阶段流失率趋势", fontsize=12, fontweight="bold")
ax3.set_ylabel("流失率")
ax3.yaxis.set_major_formatter(PercentFormatter(1.0))
ax3.grid(True, alpha=0.2, axis="y")
ax3.tick_params(axis="x", labelsize=8)

# --- 子图4（右下）：城市等级环形图 ---
ax4 = axes[1, 1]
pie_data = composition_summary
wedges, texts, autotexts = ax4.pie(
    pie_data["占比"], labels=pie_data["城市等级"],
    autopct="%1.1f%%", startangle=90,
    colors=["#3498DB", "#2ECC71", "#F39C12"],
    pctdistance=0.78,
)
for t in autotexts:
    t.set_fontsize(9)
    t.set_fontweight("bold")
    t.set_color("white")
centre = plt.Circle((0, 0), 0.55, fc="white", linewidth=0)
ax4.add_artist(centre)
ax4.set_title("用户城市等级构成", fontsize=12, fontweight="bold")

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.relative_to(ROOT))

## 综合发现与局限

1. 综合发现1：用户生命周期是流失风险的最强关联因素。新用户（Tenure=0）流失率高达53.5%，而24个月以上用户流失率为0.0%，流失率随生命周期推进呈严格单调递减。证据：折线图（03_ordered_line.png）展示了5个阶段的流失率完整趋势，各阶段样本量均大于400人（合计5,630人），趋势稳定可靠。

2. 综合发现2：低订单数和高流失率高度重叠。散点图显示流失用户（红色）高度集中于OrderCount ≤ 3的低频消费区域，而高订单数用户（OrderCount ≥ 5）几乎全部为留存用户。证据：散点图（02_behavior_scatter.png）中5,630名用户的分布清晰展示了这一聚集模式，低活跃行为可能是流失的前兆信号。

3. 综合发现3：支付方式与流失率存在差异化关联。E wallet用户流失率（约21.2%）明显高于UPI用户（约13.8%），但各支付方式的用户规模差异极大（Debit Card 2,314人 vs UPI 414人）。证据：柱状图（01_category_bar.png）同时展示了用户数和流失率的对比，揭示了规模和风险之间的权衡。

4. 数据或方法局限：（1）数据为横截面快照，无法观测同一用户的行为变化过程，流失率随生命周期递减可能是幸存者偏差；（2）`CashbackAmount`是返现金额而非消费金额，不能用于GMV分析；（3）二线城市用户仅242人（4.3%），以城市等级为分组维度的分析需注意小样本限制；（4）所有发现均为关联性观察，不能直接推断因果关系；（5）数据缺少订单金额、订单日期和用户注册日期，限制了时间维度和价值维度的分析深度。

注意：`CashbackAmount`是返现金额，不是销售额、收入或GMV。

## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
# TODO：填写5行清单，不得保留"请填写"
chart_manifest = pd.DataFrame([
    {
        "chart_id": "01",
        "file_name": "01_category_bar.png",
        "business_question": "不同支付方式的用户规模和流失率是否存在差异？哪种支付方式的用户流失风险最高？",
        "chart_type": "bar",
        "key_finding": "E wallet用户流失率最高(21.2%,614人)，UPI用户流失率最低(13.8%,414人)，Debit Card用户数最多(2,314人)。支付方式间流失率最大差异约7.4个百分点。",
        "limitation": "柱状图仅展示组间关联，不能证明支付方式导致流失。支付偏好可能与用户收入、消费习惯等混淆变量相关。",
    },
    {
        "chart_id": "02",
        "file_name": "02_behavior_scatter.png",
        "business_question": "用户的订单数量与返现金额之间是否存在关联？流失用户和留存用户在订单-返现行为上有何不同？",
        "chart_type": "scatter",
        "key_finding": "流失用户主要集中在低订单数区域(OrderCount≤3)，高订单数用户(≥5)几乎全部留存。返现金额与订单数整体呈正相关。",
        "limitation": "相关≠因果，不能推断增加返现就能降低流失。散点重叠区域难以精确判断密度差异。",
    },
    {
        "chart_id": "03",
        "file_name": "03_ordered_line.png",
        "business_question": "随着用户生命周期阶段的推进（新用户→24个月以上），流失率呈现怎样的变化趋势？",
        "chart_type": "line",
        "key_finding": "流失率从新用户53.5%严格单调递减至24个月以上0.0%，各阶段均偏离总体流失率(16.8%)。新用户流失风险是13-24个月用户的8倍以上。",
        "limitation": "横截面数据不能证明因果时间趋势，可能存在幸存者偏差。这是有序阶段比较而非时间序列分析。",
    },
    {
        "chart_id": "04",
        "file_name": "04_composition_chart.png",
        "business_question": "用户在城市等级（1/2/3级）上的分布结构是怎样的？哪类城市用户占比最大？",
        "chart_type": "pie_or_bar",
        "key_finding": "一线城市用户占65.1%(3,666人)，三线城市占30.6%(1,722人)，二线城市仅占4.3%(242人)。用户分布呈头部集中特征。",
        "limitation": "仅展示静态构成，不反映各城市用户的活跃度或流失率差异。二线城市仅242人，独立分析时样本量不足。",
    },
    {
        "chart_id": "05",
        "file_name": "day06_visualization_summary.png",
        "business_question": "整体概览",
        "chart_type": "dashboard",
        "key_finding": "综合4张图：①流失率随生命周期递减(53.5%→0.0%)；②流失用户集中在低订单数区域；③E wallet支付方式流失风险最高；④用户65%集中在一线城市。",
        "limitation": "综合图为4张子图缩略版，适合概览但细节不如独立图清晰。所有子图共享相同的横截面数据局限。",
    },
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)

In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")
